# Task 4.6 — Fraction of Variants with HAI ≥ 40 (D28)

**4.6 Predict antibody breadth - all variants (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI / Measure: Percentage / Metric: Spearman correlation
* Full description: Percentage of variants with HAI >= 40 at Day 28

---

**Target:** fraction of measured variants where raw HAI ≥ 40 (i.e., log2(HAI) ≥ log2(40) ≈ 5.32). Ranges from 0.0 to 1.0. Predictions are saved directly — no `exp2` inverse transform.

**Features:** all columns available at D0 + D7 (demographics, baseline HAI, transcriptomics). All D28 and D365 columns excluded to prevent leakage.

**CV:** 5-fold; Spearman correlation on holdout predictions.

In [ ]:
HAI_THRESHOLD = 40
AUTO_ML_MAX_RUNTIME_SECONDS = 1200

In [ ]:
PARQUET_PATH = '../merged_data/combined.parquet'
CHALLENGE_DATA_PATH = '../cleaned_data'
SUBMISSION_PATH = '../automl_submission'

In [ ]:
import io
import os
import tempfile
import warnings
from contextlib import redirect_stderr, redirect_stdout

import h2o
import numpy as np
import pandas as pd
from h2o.automl import H2OAutoML
from scipy.stats import spearmanr

warnings.filterwarnings('ignore', category=UserWarning, module='h2o')
h2o.init()

In [ ]:
data = h2o.import_file(PARQUET_PATH)
print(f'Training data shape: {data.shape}')

challenge_participants = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_participants_cleaned.csv')
challenge_hai = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_hai_cleaned.csv')
challenge_data = challenge_hai.merge(challenge_participants, on='participant_id', how='inner')
print(f'Challenge shape: {challenge_data.shape}')

In [ ]:
# Compute target in pandas and round-trip via parquet so H2O reads NaN as H2O NA
hai_d28_cols = [c for c in data.columns if c.startswith('HAI_') and c.endswith('_d28')]

pd_hai_d28 = data[hai_d28_cols].as_data_frame()
threshold_log2 = np.log2(HAI_THRESHOLD)
above = (pd_hai_d28 >= threshold_log2).sum(axis=1)
measured = pd_hai_d28.notna().sum(axis=1)
target_series = (above / measured).where(measured > 0)

_fd, _tmp = tempfile.mkstemp(suffix='.parquet')
os.close(_fd)
target_series.to_frame(name='TARGET_4_6').to_parquet(_tmp, index=False)
data = data.cbind(h2o.import_file(_tmp))
os.unlink(_tmp)

print(f'Training samples with valid target: {target_series.notna().sum()}')

---
## AutoML Training

In [ ]:
y = 'TARGET_4_6'
x = [c for c in data.columns
     if not c.endswith('_d28') and not c.endswith('_d365')
     and c != 'participant_id' and c != y]

train = data[data[y].isna() == 0]
print(f'Training samples: {train.nrows}  |  Features: {len(x)}')

aml = H2OAutoML(max_models=10, seed=1, nfolds=5,
                keep_cross_validation_predictions=True,
                max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml.train(x=x, y=y, training_frame=train)
print('Training complete.')

In [ ]:
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
cv_preds = aml.leader.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train[y].as_data_frame()[y]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.6 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')

In [ ]:
print(f'Leader model: {aml.leader.model_id}')
varimp = aml.leader.varimp(use_pandas=True)
display(varimp.head(20))
aml.leader.varimp_plot(num_of_features=20)

In [ ]:
challenge_hf = h2o.H2OFrame(challenge_data)
y_pred = aml.leader.predict(challenge_hf).as_data_frame()['predict']

results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.6': y_pred,  # fraction — no inverse transform
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_6.csv', index=False)
results

In [ ]:
h2o.cluster().shutdown()

---
## Conclusion

- **Leader model:** (fill after run)
- **CV Spearman:** (fill after run)

**Target:** fraction of measured HAI variants where raw titer ≥ 40 at D28. Unlike Tasks 4.4/4.5, this is a proportion (not log2 HAI), so predictions are saved directly without `exp2`.

Submission saved to `automl_submission/task_4_6.csv`.